# TensorFlow / Keras Pipeline -- Sonar Classification

In this exercise we build an end-to-end binary classification pipeline with the **Keras 3** API on the **TensorFlow** backend. It is the Keras alternative to the PyTorch Sonar lab.

**Dataset:** `sonar.all-data.csv` -- 208 samples, 60 continuous sonar features, label column is `R` (rock) or `M` (mine).

**Evaluation protocol (as in the PyTorch lab):** three partitions are made before any scaling -- 124 training, 42 validation, and 42 test rows, with the same label mapping (Rock = 0, Mine = 1), split calls, and seed as the PyTorch notebook. The scaler is fitted on the training rows only. Early stopping and the architecture comparison use the validation rows; the test rows are used once, at the end, for the chosen model.

**Requirements:** TensorFlow 2.16 or later, which installs Keras 3 (for example `pip install "tensorflow>=2.16,<3"`, in a separate environment if it conflicts with PyTorch).

**Learning objectives:**
1. Build a Keras `Sequential` model and compile it with a loss that matches its output layer.
2. Use `model.fit()` with `EarlyStopping` and an explicit `validation_data` set.
3. Plot Keras training history (loss and accuracy curves).
4. Compare architectures on validation data, then evaluate the chosen model once on the test set.

---
## 1. Imports

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")   # must be set before importing keras

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
print(f"Keras {keras.__version__}, backend: {keras.config.backend()}")

---
## 2. Load and Explore the Data

The CSV has **no header row**. There are 60 feature columns followed by a label column (`R` or `M`).

In [ ]:
# YOUR CODE HERE
# Hint: use pd.read_csv with header=None, create column names feat_0..feat_59, label


In [ ]:
# YOUR CODE HERE
# Hint: df["label"].value_counts()


---
## 3. Encode Labels, Split, and Standardize

We map the labels explicitly (Rock = 0, Mine = 1, the positive class) and print the mapping. Then we make three partitions **before** fitting the scaler, with the same calls and seed as the PyTorch lab (124 training, 42 validation, 42 test rows), and fit the scaler on the training rows only. Keras's `validation_split` is not used: it would take the *last* rows of the array without shuffling, after the scaler had already seen them.

**Tasks:**
- Map the labels with `LABEL_MAP = {"R": 0, "M": 1}` (float32) and print the mapping and class counts
- Split row indices: first 20% test (`stratify=y`, `random_state=SEED`), then 25% of the rest as validation
- Fit `StandardScaler` on the training rows only; transform training, validation, and test rows

In [ ]:
# YOUR CODE HERE
# Hint: LABEL_MAP = {"R": 0, "M": 1}; y = df["label"].map(LABEL_MAP).to_numpy(dtype=np.float32)
#       X = df.drop(columns=["label"]).to_numpy(dtype=np.float32)

In [ ]:
# YOUR CODE HERE
# Hint: dev_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED)
#       train_idx, val_idx = train_test_split(dev_idx, test_size=0.25, stratify=y[dev_idx], random_state=SEED)
# Then StandardScaler: fit on X[train_idx] only, transform the training, validation and test rows

---
## 4. Build the Keras Sequential Model

A two-hidden-layer network (the PyTorch lab uses one hidden layer of 32 units; translating that exact model is an extension):
- Dense(128, relu) + Dropout(0.3)
- Dense(64, relu) + Dropout(0.3)
- Dense(1, sigmoid): the output is a probability, not a logit

Call `keras.utils.set_random_seed(SEED)` first, and start the model with `keras.Input(shape=(60,))`.

In [ ]:
# YOUR CODE HERE
# Hint:
#   keras.utils.set_random_seed(SEED)
#   model = keras.Sequential([
#       keras.Input(shape=(60,)),
#       layers.Dense(128, activation="relu"),
#       layers.Dropout(0.3),
#       ...
#   ])
#   model.summary()

---
## 5. Compile the Model

Use the Adam optimizer with `learning_rate=1e-3`, binary cross-entropy, and track `accuracy`. Which `from_logits` setting matches the sigmoid output layer?

In [ ]:
# YOUR CODE HERE
# Hint: model.compile(optimizer=..., loss=..., metrics=[...])


---
## 6. Train with EarlyStopping

Use `callbacks.EarlyStopping` monitoring `val_loss` with `patience=15` and `restore_best_weights=True`.
Train for up to 200 epochs with `batch_size=16`, passing the validation rows with `validation_data=(X_val, y_val)` (not `validation_split`).

In [ ]:
# YOUR CODE HERE
# Hint:
#   early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
#   history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=200,
#                       batch_size=16, callbacks=[early_stop], verbose=1)

---
## 7. Plot Training History

Create two side-by-side plots: **Loss** (train vs val) and **Accuracy** (train vs val).

In [ ]:
# YOUR CODE HERE
# Hint: use history.history["loss"], history.history["val_loss"],
#       history.history["accuracy"], history.history["val_accuracy"]


---
## 8. Evaluate on the Validation Set

Diagnose the restored model on the validation rows: loss, accuracy, classification report, and confusion matrix. These rows also chose the restored epoch, so the scores are slightly optimistic. With 42 rows, one row moves accuracy by 2.4 points.

Use `model.evaluate()` for loss and accuracy, then `model.predict()` and a 0.5 threshold on the predicted probabilities for a classification report.

In [ ]:
# YOUR CODE HERE
# Hint:
#   val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
#   val_prob = model.predict(X_val, verbose=0).ravel()
#   val_pred = (val_prob >= 0.5).astype(int)
#   print(classification_report(y_val, val_pred, target_names=CLASS_NAMES))

---
## 9. Compare Architectures on the Validation Set

Write a helper function `build_model(hidden_units, dropout_rate, lr)` that returns a compiled model.
Train at least 3 different architectures with `EarlyStopping` on the validation rows, compare their validation loss and accuracy, and choose one (lowest validation loss). Do not use the test set.

In [ ]:
# YOUR CODE HERE
# Hint: define build_model(), loop over an architectures list,
# train each with EarlyStopping on validation_data=(X_val, y_val), record validation loss/accuracy,
# and keep the fitted model with the lowest validation loss as final_model

---
## 10. One Locked Final Assessment

The architecture was chosen on the validation rows. Evaluate the chosen model **once** on the 42 test rows and report the score with its count. Do not change the architecture because of this result: that would turn the test set into a second validation set.

Use `final_model.evaluate()` on the test rows, then `final_model.predict()` with a 0.5 threshold for a classification report and confusion matrix.

In [ ]:
# YOUR CODE HERE
# Hint:
#   test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)   # once
#   y_prob = final_model.predict(X_test, verbose=0).ravel()
#   y_pred = (y_prob >= 0.5).astype(int)

---
## 11. Summary

In this notebook we:

1. Loaded the Sonar dataset and mapped the labels explicitly (Rock = 0, Mine = 1).
2. Made training, validation, and test partitions before scaling, as in the PyTorch lab, and fitted the scaler on the training rows only.
3. Built a Keras `Sequential` model with two hidden layers and a sigmoid output.
4. Compiled it with Adam and the binary cross-entropy setting that matches a sigmoid output.
5. Trained with `model.fit()`, explicit `validation_data`, and `EarlyStopping` with restored best weights.
6. Plotted training history (loss and accuracy curves).
7. Compared architectures on the validation rows and evaluated the chosen one once on the test rows.

**Key takeaways:**
- Keras provides a high-level API that abstracts away the training loop.
- Pass explicit `validation_data`; `validation_split` takes the last rows unshuffled, after preprocessing has already seen them.
- `EarlyStopping(restore_best_weights=True)` is Keras's copied checkpoint.
- Choose with validation data, then touch the test set once.